In [1]:
from dataset import *

In [1]:
img_directory = '/datacommons/carlsonlab/srs108/blooms/downloads'
lbl_directory = '/datacommons/carlsonlab/srs108/blooms/lbls'
imgpaths = []

for file in sorted(listdir(img_directory)):
    if file.endswith('.jpg'):
        imgpaths.append(os.path.join(img_directory,file))
lblpaths = []
for file in sorted(listdir(lbl_directory)):
    if file.endswith('.jpg'):
        lblpaths.append(os.path.join(lbl_directory,file))
        
value = random.randint(0, 100)

with rio.open(imgpaths[value]) as rds:
    data = rds.read()
    img_arr = np.array(data)
    imgdata = torch.from_numpy(img_arr.copy())
    
    
with rio.open(lblpaths[value]) as rds:
    data = rds.read()
#     data = np.moveaxis(data, 0, 2) 
    fpt_arr = np.array(data)
    fptdata =  torch.from_numpy(fpt_arr.copy())
    print(fptdata.shape)
    
crop_height, crop_width = 320, 320

height, width = imgdata.shape[1:]

# Randomly select the top left corner of the crop
x = np.random.randint(0, width - crop_width + 1)
y = np.random.randint(0, height - crop_height + 1)

# Perform the crop
imgdata = imgdata[:, y:y + crop_height, x:x + crop_width]  
fptdata = fptdata[:, y:y + crop_height, x:x + crop_width]  


mirror = np.random.randint(0, 2)
flip = np.random.randint(0, 2)
if mirror:
    print('horizontal flip')
    imgdata = np.fliplr(imgdata)
    fptdata = np.fliplr(fptdata)
    
if flip:
    print('vertical flip')
    imgdata = np.flipud(imgdata)
    fptdata = np.flipud(fptdata)
    
##rotate by [0,1,2,3]*90 deg
rot = np.random.randint(0, 4)
print(f'rotated {rot* 90} degrees ')
imgdata = np.rot90(imgdata, rot, axes=(1,2))
fptdata = np.rot90(fptdata, rot, axes=(1,2))

min_class = 100
max_class = 249
label = fptdata

log_label = np.log1p(label)
log_threshold1 = np.log1p(30)  # Corresponding to your original threshold 30
log_threshold2 = np.log1p(100) # Corresponding to your original threshold 100
log_threshold3 = np.log1p(249) # Corresponding to your original threshold 249

# Initialize the new_label array
new_label = np.full_like(label, 0)

# Bin the log-transformed pixel values based on these thresholds
new_label[log_label < log_threshold1] = 1
new_label[(log_label >= log_threshold1) & (log_label <= log_threshold2)] = 2
new_label[(log_label > log_threshold2) & (log_label <= log_threshold3)] = 3


mask = (label >= min_class) & (label <= max_class)

new_label2 = np.full_like(label, 0)

new_label2[fptdata < 31] = 1
new_label2[(fptdata >= 31) & (fptdata <= 100)] = 2
new_label2[(fptdata > 100) & (fptdata <= 249)] = 3
# num_classes = len(np.unique(new_label))

# # mapped_labels = np.clip(np.array([map_value(val) for val in new_label.flatten()]), 0, 10) 
# mapped = mapped_labels.reshape(320,320,1)

fig,ax = plt.subplots(2,2, figsize=(15,12))
imgdata = np.moveaxis(imgdata, 0, 2) 
fptdata = np.moveaxis(fptdata, 0,2)
new_label = np.moveaxis(new_label, 0,2)
new_label2 = np.moveaxis(new_label2, 0,2)
ax[0,0].imshow(imgdata)
ax[0,0].set_title('RGB Image')
ax[0,0].axis('off')
ax[0,1].set_title('Label Image')
ax[0,1].imshow(fptdata)
ax[0,1].axis('off')
ax[1,0].set_title('logBinned 0-30, 31-100, 101-249')
ax[1,0].imshow(new_label)
ax[1,0].axis('off')
ax[1,1].set_title('DN 100-249 binned in 10')
ax[1,1].imshow(new_label2)
ax[1,1].axis('off')
plt.show()
print(np.unique(new_label))
# print(np.unique(new_label))

In [6]:
# np.unique(fptdata)
colors, counts = np.unique(fptdata, axis=0, return_counts=True)


In [2]:
im = fptdata.ravel()
pixel_counts = np.bincount(im, minlength=256)

# To see the counts of each pixel value
for pixel_value, count in enumerate(pixel_counts):
    if count > 0:  # Only print pixel values that occur in the image
        print(f"Pixel value {pixel_value}: {count} counts")

In [2]:
df = pd.read_csv('CI_Cyano_Color_Scale.csv')

In [3]:
Pixel Value (DN)=np.round(83.3×(log10​(CI)+4.2))

,Index (DN),Red,Green,Blue,Color,CI,Cells/ml x 1000\nApprox,Meaning,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,0,0,0,0,NaN,0.000063,6.309573,No data,NaN,NaN,NaN,Pixel Value (DN)=np.round(83.3×(log10​(CI)+4.2))
1,1,72,25,107,NaN,0.000065,6.486344,NaN,NaN,NaN,NaN,NaN
2,2,72,26,108,NaN,0.000067,6.668068,NaN,NaN,NaN,NaN,NaN
3,3,72,28,110,NaN,0.000069,6.854882,NaN,NaN,NaN,NaN,CI=10(250.03.0​×DN−4.2)
4,4,72,30,112,NaN,0.000070,7.046931,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
251,251,82,82,82,NaN,NaN,NaN,Adjacency,NaN,NaN,NaN,NaN
252,252,247,247,247,NaN,NaN,NaN,Land,NaN,NaN,NaN,NaN
253,253,204,204,204,NaN,NaN,NaN,Cloud,NaN,NaN,NaN,NaN
254,254,150,150,150,NaN,NaN,NaN,Invalid or mixed,NaN,NaN,NaN,NaN


In [ ]:
markers = set()
imgpaths = []
label_paths = {}

for file in sorted(listdir('/datacommons/carlsonlab/srs108/crops')):
    parts = file.split('_')[0]  
    if parts not in markers:
        imgpaths.append(os.path.join('/datacommons/carlsonlab/srs108/crops', file))
        markers.add(parts)



for folder in sorted(os.listdir('/datacommons/carlsonlab/srs108/blooms/all_data')):
    if folder not in [".DS_Store", ".ipynb_checkpoints"]:
        folder_path = os.path.join('/datacommons/carlsonlab/srs108/blooms/all_data', folder)
        for file in sorted(os.listdir(folder_path)):
            if file.endswith('.tif'):
                parts = file.split('.')
                name = parts[-3] 
                marker = parts[1][:4] + parts[2]  
                if marker in markers and name == 'CIcyano':
                    if marker not in label_paths: 
                        label_paths[marker] = os.path.join(folder_path, file)
                    
matches = {}
for img_path in imgpaths:
    img_marker = img_path.split('/')[-1].split('_')[0]  
    if img_marker in markers:
        matches[img_marker] = {
            'image': img_path,
            'label': label_paths.get(img_marker, None)
        }

In [ ]:
# matches = {}
# for img_path in imgpaths:
#     img_marker = img_path.split('/')[-1].split('_')[0]  
#     if img_marker in markers:
#         matches[img_marker] = {
#             'image': img_path,
#             'label': label_paths.get(img_marker, None)
#         }
# keys = list(matches.keys())
# marler = keys[0]
# with rio.open(matches[marker]['image']) as rds:

In [ ]:
matches = {marker: paths for marker, paths in matches.items() if paths['label'] is not None}
len(matches)

In [ ]:
img_dir = '/datacommons/carlsonlab/srs108/crops/chowan_river/'
for file in sorted(os.listdir(img_dir)):
    with rio.open(os.path.join(img_dir,file)) as rds:
        data = rds.read()
#         data = np.moveaxis(data, 0, 2) 
        imgdata = np.array(data)
        h = np.moveaxis(imgdata, 0,2)
        plt.imshow(h)
        plt.show()
        print(imgdata.shape)
        crop_height, crop_width = 224, 224
        
        height, width = imgdata.shape[1:]

        x = np.random.randint(0, width - crop_width + 1)
        y = np.random.randint(0, height - crop_height + 1)
        print(x,y)
        # Perform the crop
        imgdata = imgdata[:, y:y + crop_height, x:x + crop_width] 
        imgdata = np.moveaxis(imgdata, 0, 2) 
        plt.imshow(imgdata)
        plt.show()

In [ ]:
###weak supervision training loop first attempt, prior to classification 
# def train(model, pool, dataloader, criterion, opt, epoch, device):
#     running_acc = 0
#     running_loss = 0
#     train_ious = []
#     model.train()

#     for i, batch in tqdm(enumerate(dataloader)):
#         x = batch['img'].float().to(device)
#         y = batch['fpt'].float().squeeze(dim=1).to(device) #add dim 1
#         output = model(x)
        
#         reduced_output = pool(output)
#         reduced_output = reduced_output.squeeze(3)  

#         # Expand the target tensor to match the number of classes
#         y = y.expand(-1, reduced_output.size(1), -1)  
        
#         output_binary = np.zeros(output.shape)
#         output_binary[output.cpu().detach().numpy() >= 0] = 1
        
# #         print(reduced_output.shape)
# #         print(y.shape)
#         #loss
#         loss = criterion(reduced_output,y)
#         running_loss += loss.item()
        
#         #mIoU
# #         total_iou, iou_list = mIOU(y, output, num_classes=4)
# #         train_ious.append(total_iou)
        
#         #pixel accuracy
# #         acc_epoch = pixel_accuracy(output, y)
# #         running_acc += acc_epoch.item()
        
#         opt.zero_grad()
#         loss.backward()
#         opt.step()
#         torch.cuda.empty_cache()
#     return running_loss/len(dataloader) #, running_acc/len(dataloader), np.average(train_ious)

In [56]:
img_dir = '/datacommons/carlsonlab/srs108/crops/mt_gould/'
lbl_dir = '/datacommons/carlsonlab/srs108/blooms/all_data'
lat,lon =  36.125495, -76.740141

markers = set()
imgpaths = []
label_paths = {}
nasa_paths = {}
for file in sorted(listdir(img_dir)):
    parts = file.split('_')[0]  
    if parts not in markers: #prevents duplicates
        imgpaths.append(os.path.join(img_dir, file))
        markers.add(parts)


for folder in sorted(os.listdir(lbl_dir)):
    if folder not in [".DS_Store", ".ipynb_checkpoints"]:
        folder_path = os.path.join(lbl_dir, folder)
        for file in sorted(os.listdir(folder_path)):
            if file.endswith('.tif'):
                parts = file.split('.')
                name = parts[-3] 
                marker = parts[1][:4] + parts[2]  
                if marker in markers and name == 'CIcyano':
                    if marker not in label_paths: 
                        label_paths[marker] = os.path.join(folder_path, file)

                        
for folder in sorted(os.listdir(lbl_dir)):
    if folder not in [".DS_Store", ".ipynb_checkpoints"]:
        folder_path = os.path.join(lbl_dir, folder)
        for file in sorted(os.listdir(folder_path)):
            if file.endswith('.tif'):
                parts = file.split('.')
                name = parts[-3] 
                marker = parts[1][:4] + parts[2]  
                if marker in markers and name == 'truecolor':
                    if marker not in nasa_paths: 
                        nasa_paths[marker] = os.path.join(folder_path, file)


matches = {}
for img_path in imgpaths:
    img_marker = img_path.split('/')[-1].split('_')[0]  
    if img_marker in markers:
        matches[img_marker] = {
            'planet': img_path,
            'cicyano': label_paths.get(img_marker, None),
            'sentinel': nasa_paths.get(img_marker,None)
        }

matches = {marker: paths for marker, paths in matches.items() if paths['cicyano'] is not None}
keys = list(matches.keys())


In [5]:
value = random.randint(0,700)
# print(value)
marker = keys[196]
with rio.open(matches[marker]['planet']) as rds:
    data = rds.read([1,2,3])
    planet_data = np.array(data) #shape (3,600,600)

with rio.open(matches[marker]['sentinel']) as rds:
    data = rds.read()
    crop_size = 2.5

    transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)
    xx, yy = transformer.transform(lon, lat)
    row, col = rds.index(xx, yy)

    # Define the crop window
    row_start = max(row - crop_size, 0)
    row_end = min(row + crop_size, rds.height)
    col_start = max(col - crop_size, 0)
    col_end = min(col + crop_size, rds.width)

    try:
        window = Window.from_slices((row_start, row_end), (col_start, col_end))
        data = rds.read(window=window)
        data = np.moveaxis(data, 0, 2)  
        sentinel_data = np.array(data)
    
    except Exception as e:
        print(f"Error processing {matches[marker]} label: {e}")
        pass
        
        
with rio.open(matches[marker]['cicyano']) as rds:
    data = rds.read()
    crop_size = 2.5

    transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)
    xx, yy = transformer.transform(lon, lat)
    row, col = rds.index(xx, yy)

    # Define the crop window
    row_start = max(row - crop_size, 0)
    row_end = min(row + crop_size, rds.height)
    col_start = max(col - crop_size, 0)
    col_end = min(col + crop_size, rds.width)

    try:
        window = Window.from_slices((row_start, row_end), (col_start, col_end))
        data = rds.read(window=window)
        data = np.moveaxis(data, 0, 2)  
        lbl_arr = np.array(data)
#         print(lbl_arr)
        fpt_avg = np.where((31 <= lbl_arr) & (lbl_arr<= 249), lbl_arr, 0) #bloom pixel avg only
#         print(fpt_avg)
        fpt_avg = fpt_avg[fpt_avg !=0]
        mean_value = fpt_avg.mean() if fpt_avg.size > 0 else 0
        #2 is land, 3 is cloud, 1 is bloom, 0 is else
        fptdata = np.where(lbl_arr == 252, 2, np.where(lbl_arr == 253, 3, np.where((31 <= lbl_arr) & (lbl_arr<= 249), 1, 0)))
        
#         print(fptdata)
        
        if 31 <= mean_value <= 249: #avg nonzero and non-land values in cicyano, if between 100-249, classify image as bloom, else not
            class_lbl = 1
        else: class_lbl = 0

    except Exception as e:
        print(f"Error processing {matches[marker]} label: {e}")
        pass
    
sample = {'planet': planet_data,
          'planet_file': matches[marker]['planet'],
          'sentinel': sentinel_data,
          'sentinel_file':matches[marker]['sentinel'],
          'cicyano': fptdata,
          'cicyano_file': matches[marker]['cicyano'],
          'ci_avg': round(mean_value),
          'class_label': class_lbl
         }

planet = sample['planet']
cicyano = sample['cicyano']
sentinel = sample['sentinel']
label = sample['ci_avg']
class_label = sample['class_label']
inf = sample['planet_file'].split('/')


planet = torch.from_numpy(planet.copy())
cicyano = torch.from_numpy(cicyano.copy())
sentinel = torch.from_numpy(sentinel.copy())
d=inf[-1].split('_')[0]
d = d[4:6]+'-'+d[6:]+'-'+d[:4]
p = inf[-2]

colors = ['black', 'blue', 'yellow', 'grey']  # Set your desired colors for each class here
cmap = ListedColormap(colors)
boundaries = [0, 1, 2, 3, 4]  # Each boundary represents a class (0, 1, 2, 3)
norm = BoundaryNorm(boundaries, cmap.N)

fig,(ax1, ax2,ax3) = plt.subplots(1,3, figsize=(9,9))
fig.suptitle(f'Date: {d} at {p}',y=0.7)
ax1.imshow(planet.permute(1,2,0), zorder=1)
ax1.axis('off')
ax1.set_title(f'Planet')
grid_color = 'white'
linewidth = 0.1
for i in range(1, 5):
    # Draw vertical lines
    ax1.vlines(x=planet.shape[1] * i / 5, ymin=0, ymax=planet.shape[0], color=grid_color, linewidth=linewidth)
    # Draw horizontal lines
    ax1.hlines(y=planet.shape[0] * i / 5, xmin=0, xmax=planet.shape[1], color=grid_color, linewidth=linewidth)


ax2.imshow(cicyano, cmap=cmap,norm=norm)
ax2.axis('off')
ax2.set_title(f'Class Label: {class_label}')

ax3.imshow(sentinel)
ax3.axis('off')

ax3.set_title(f'Sentinel Avg Label: {label}')
plt.show()

In [22]:
value = random.randint(0,700)

marker = keys[value]
with rio.open(matches[marker]['planet']) as rds:
    data = rds.read([1,2,3])
    planet_data = np.array(data) #shape (3,600,600)
    planet_data = np.moveaxis(planet_data, 0,2)
    
with rio.open(matches[marker]['sentinel']) as rds:
    data = rds.read()
    crop_size = 2.5

    transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)
    xx, yy = transformer.transform(lon, lat)
    row, col = rds.index(xx, yy)

    # Define the crop window
    row_start = max(row - crop_size, 0)
    row_end = min(row + crop_size, rds.height)
    col_start = max(col - crop_size, 0)
    col_end = min(col + crop_size, rds.width)

    try:
        window = Window.from_slices((row_start, row_end), (col_start, col_end))
        data = rds.read(window=window)
        data = np.moveaxis(data, 0, 2)  
        sentinel_data = np.array(data)
    
    except Exception as e:
        print(f"Error processing {matches[marker]} label: {e}")
        pass
        
        
with rio.open(matches[marker]['cicyano']) as rds:
    data = rds.read()
    crop_size = 2.5

    transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)
    xx, yy = transformer.transform(lon, lat)
    row, col = rds.index(xx, yy)

    # Define the crop window
    row_start = max(row - crop_size, 0)
    row_end = min(row + crop_size, rds.height)
    col_start = max(col - crop_size, 0)
    col_end = min(col + crop_size, rds.width)

    try:
        window = Window.from_slices((row_start, row_end), (col_start, col_end))
        data = rds.read(window=window)
        data = np.moveaxis(data, 0, 2)  
        lbl_arr = np.array(data)

        fpt_avg = np.where((100 <= lbl_arr) & (lbl_arr<= 249), lbl_arr, 0) #bloom pixel avg only
        fpt_avg = fpt_avg[fpt_avg !=0]
        mean_value = fpt_avg.mean() if fpt_avg.size > 0 else 0
        #2 is land, 3 is cloud, 1 is bloom, 0 is else
        fptdata = np.where(lbl_arr == 252, 2, np.where(lbl_arr == 253, 3, np.where((100 <= lbl_arr) & (lbl_arr<= 249), 1, 0)))

#         print(fptdata)
        
        if 31 <= mean_value <= 249: #avg nonzero and non-land values in cicyano, if between 100-249, classify image as bloom, else not
            class_lbl = 1
        else: class_lbl = 0

    except Exception as e:
        print(f"Error processing {matches[marker]} label: {e}")
        pass
    
sample = {'planet': planet_data,
          'planet_file': matches[marker]['planet'],
          'sentinel': sentinel_data,
          'sentinel_file':matches[marker]['sentinel'],
          'cicyano': fptdata,
          'cicyano_file': matches[marker]['cicyano'],
          'ci_avg': round(mean_value),
          'class_label': class_lbl
         }


# planet = sample['planet']
# cicyano = sample['cicyano']
# sentinel = sample['sentinel'] 


# fig, ax = plt.subplots(1,3, figsize=(9,9))
# ax[0].imshow(planet)
# ax[0].axis('off')

# colors = ['black', 'seagreen', 'yellow', 'grey']  # Set your desired colors for each class here
# cmap = ListedColormap(colors)
# boundaries = [0, 1, 2, 3, 4]  # Each boundary represents a class (0, 1, 2, 3)
# norm = BoundaryNorm(boundaries, cmap.N)
# ax[1].imshow(cicyano, cmap=cmap,norm=norm)
# ax[1].axis('off')

# ax[2].imshow(sentinel)
# ax[2].axis('off')
# plt.show()

# mirror = np.random.randint(0, 2)
# flip = np.random.randint(0, 2)
# if mirror:
#     print('Horizontal Flip')
#     planet = np.fliplr(planet)
#     cicyano = np.fliplr(cicyano)
#     sentinel = np.fliplr(sentinel)
#     fig, ax = plt.subplots(1,3, figsize=(9,9))
#     ax[0].imshow(planet)
#     ax[0].axis('off')

#     ax[1].imshow(cicyano, cmap=cmap,norm=norm)
#     ax[1].axis('off')

#     ax[2].imshow(sentinel)
#     ax[2].axis('off')

#     plt.show()
# if flip:
#     print('Vertical Flip')
#     planet = np.flipud(planet)
#     cicyano = np.flipud(cicyano)
#     sentinel = np.flipud(sentinel)
#     fig, ax = plt.subplots(1,3, figsize=(9,9))
#     ax[0].imshow(planet)
#     ax[0].axis('off')

#     ax[1].imshow(cicyano, cmap=cmap,norm=norm)
#     ax[1].axis('off')

#     ax[2].imshow(sentinel)
#     ax[2].axis('off')

#     plt.show()
    

# # rotate by [0,1,2,3]*90 deg
# rot = np.random.randint(0, 4)
# planet = np.rot90(planet, rot, axes=(0,1))
# cicyano = np.rot90(cicyano, rot, axes=(0,1))
# sentinel = np.rot90(sentinel, rot, axes=(0,1))



# print(f'rotated {rot* 90} degrees: {planet.shape, cicyano.shape, sentinel.shape}')

# fig, ax = plt.subplots(1,3, figsize=(9,9))
# ax[0].imshow(planet)
# ax[0].axis('off')

# ax[1].imshow(cicyano, cmap=cmap,norm=norm)
# ax[1].axis('off')

# ax[2].imshow(sentinel)
# ax[2].axis('off')

# plt.show()

In [28]:
g = '/datacommons/carlsonlab/srs108/crops/cannons_ferry/20170125_150618_0e19.tif'

In [3]:
# import numpy as np
# import matplotlib.pyplot as plt
# import rasterio
# import os

# dirr = '/datacommons/carlsonlab/srs108/crops/chowan_river/'

# # Ensure that you import listdir from os
# from os import listdir

# for file in sorted(listdir(dirr)):
#     print(file)

#     with rasterio.open(os.path.join(dirr, file)) as src:
#         # Read bands 1, 2, and 3 into a 3D array, then reorder the axes
#         data = src.read([1, 2, 3])  # Assuming 1, 2, and 3 are RGB bands
#         data = np.moveaxis(data, 0, 2)

#         # Read individual bands for SS and CI calculation
#         green_band = src.read(2)  # Green band (approx 550 nm)
#         red_band = src.read(3)    # Red band (approx 630 nm)
#         nir_band = src.read(4)    # NIR band (approx 820 nm)

#     # Define approximate central wavelengths for each band
#     lambda1 = 550  # Green
#     lambda2 = 630  # Red (as a rough substitute for 680 nm)
#     lambda3 = 820  # NIR

#     # Calculate SS (Cyanobacteria Index)
#     SS = red_band - green_band + ((green_band - nir_band) * (lambda2 - lambda1) / (lambda3 - lambda1))

#     # Calculate CI (Cyanobacteria Index)
#     CI = -SS
#     print(np.mean(CI))  # Print the mean of the CI

#     # Extract the date from the filename (assuming it's in the first 8 characters)
#     t = file[:8]

#     # Create subplots to display both CI and the RGB image
#     f, a = plt.subplots(1, 2, figsize=(9, 9))

#     # Plot CI in the first subplot
#     im = a[0].imshow(CI)
#     f.colorbar(im, ax=a[0], label='Cyanobacteria Index (CI)', shrink=0.3)
#     a[0].set_title(f'Approximate Cyanobacteria Index')
#     a[0].axis('off')  # Hide axis on CI plot

#     # Plot the RGB image in the second subplot
#     a[1].imshow(data)
#     a[1].set_title(t)
#     a[1].axis('off')  # Hide axis on the image plot

#     plt.show()


In [10]:
import rasterio
from rasterio.warp import transform


In [4]:
img_dir = '/datacommons/carlsonlab/srs108/crops/chowan_river/'
lbl_dir = '/datacommons/carlsonlab/srs108/blooms/all_data'
# lat,lon =  36.055233, -76.686425

markers = set()
imgpaths = []
label_paths = {}
nasa_paths = {}
for file in sorted(listdir(img_dir)):

#     print(file)
        
    with rasterio.open(os.path.join(img_dir,file)) as dataset:
        z = dataset.read()[0]
        print(dataset.crs)
#         # Get the affine transformation matrix
#         transform_matrix = dataset.transform

#         # Get the pixel coordinates (e.g., center of the image)
#         pixel_x = dataset.width // 2
#         pixel_y = dataset.height // 2

#         # Convert pixel coordinates to geographic coordinates in the image's CRS
#         x, y = rasterio.transform.xy(transform_matrix, pixel_y, pixel_x)

#         # Get the CRS of the dataset
#         src_crs = dataset.crs  # Source CRS of the image

#         # Transform to latitude and longitude (EPSG:4326)
#         lon, lat = transform([src_crs], ['EPSG:4326'], [x], [y])
        
#         # Output latitude and longitude
#         print(f"Coordinates: ({lat[0]:.6f}, {lon[0]:.6f})")